<a href="https://colab.research.google.com/github/mc-castro/clinicaltrials-ia-thesis/blob/mc-castro%2Fdevelop/notebooks/04_clinical_patients_match.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
import pandas as pd
import ast
from google.colab import userdata, drive
from tqdm import tqdm
import typing_extensions as typing
import google.generativeai as genai
import json
import time
import os
import csv
from datetime import datetime, timedelta


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [ ]:
# 1. Load the data and convert strings back to real Python lists
clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")
# clinical_trials_df['inclusion_criteria'] = clinical_trials_df['inclusion_criteria'].apply(ast.literal_eval)

In [ ]:
mimic_patients_df = pd.read_parquet("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet")
print(mimic_patients_df.columns.tolist())

['subject_id', 'hadm_id', 'HPI', 'physical_exam', 'chief_complaint', 'icd_diag_principal', 'name_diag_principal', 'list_procedures', '% Hemoglobin A1c', '25-OH Vitamin D', 'ADP', 'ANCA Titer', 'ARCH-1', 'Absolute Basophil Count', 'Absolute CD3 Count', 'Absolute CD4 Count', 'Absolute CD8 Count', 'Absolute Eosinophil Count', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 'Absolute Neutrophil Count', 'Acanthocytes', 'Acetaminophen', 'Acetone', 'Alanine Aminotransferase (ALT)', 'Albumin', 'Alkaline Phosphatase', 'Alpha-Fetoprotein', 'Alveolar-arterial Gradient', 'Ammonia', 'Amylase', 'Anion Gap', 'Anisocytosis', 'Anti-Mitochondrial Antibody', 'Anti-Neutrophil Cytoplasmic Antibody', 'Anti-Nuclear Antibody', 'Anti-Nuclear Antibody, Titer', 'Anti-Smooth Muscle Antibody', 'Anti-Thyroglobulin Antibodies', 'Anticardiolipin Antibody IgG', 'Anticardiolipin Antibody IgM', 'Antithrombin', 'Arachadonic Acid', 'Asparate Aminotransferase (AST)', 'Assist/Control', 'Atypical Lymphocytes', 'Bands

In [ ]:
len(mimic_patients_df)

2813

## Transform

In [ ]:
# @title Clinical Match Engine

class BatchClinicalMatch:
    def __init__(self, api_key, version_genai):
        genai.configure(api_key=userdata.get(api_key))
        self.model = genai.GenerativeModel(version_genai)

    def _process_batch(self, batch_df, inclusion, exclusion):
        # Transformamos o lote de pacientes em uma lista compacta de JSON
        # patients_list = batch_df.to_json(orient='records')

        # 1. Converte o lote para uma lista de dicionários
        raw_patients = batch_df.to_dict(orient='records')

        # 2. LIMPEZA DINÂMICA: Remove campos nulos/vazios de cada paciente
        compact_patients = []
        for p in raw_patients:
            # Mantém apenas o que não é nulo, não é NaN e não é string vazia
            clean_p = {k: v for k, v in p.items() if pd.notna(v) and str(v).strip() != "" and str(v).lower() != "nan"}
            compact_patients.append(clean_p)

        # 3. Transforma a lista limpa em string JSON
        patients_json_string = json.dumps(compact_patients)

        prompt = f"""
        As a clinical trial coordinator, analyze this batch of patient records against the criteria.

        STUDY CRITERIA:
        Inclusion: {inclusion}
        Exclusion: {exclusion}

        PATIENTS LIST (JSON):
        {patients_json_string}

        For each patient, decide if they are ELIGIBLE.
        Consider synonyms (e.g., 'Synthetic Substitute' = 'Prosthesis').
        'No pregnant women' = Males OK, non-pregnant Females OK.

        RETURN ONLY A JSON LIST of eligible patients in this format:
        [
          {{"subject_id": 123, "reason": "Short reason"}},
          {{"subject_id": 456, "reason": "Short reason"}}
        ]
        If NO patients are eligible, return an empty list [].
        """
        try:
            response = self.model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
            return json.loads(response.text)
        except:
            return []

    def run_inference(self, studies_df, patients_df, batch_size=50, report_file="matching_report.csv", log_file="audit_log.csv"):
        # --- LÓGICA DE CHECKPOINT ---
        processed_ncts = set()
        if os.path.exists(report_file):
            existing_report = pd.read_csv(report_file)
            processed_ncts = set(existing_report['nct_id'].unique())
            print(f"Retomando progresso: {len(processed_ncts)} estudos já processados.")

        for _, study in tqdm(studies_df.iterrows(), total=len(studies_df), desc="Estudos"):
            nct_id = study['nct_id']

            # Pula se já estiver no arquivo
            if nct_id in processed_ncts:
                continue

            study_eligible_ids = []
            audit_data = []

            for i in range(0, len(patients_df), batch_size):
                print(f" -> Processando lote {i//batch_size + 1} de {len(patients_df)//batch_size + 1}...", end="\r")
                batch = patients_df.iloc[i : i + batch_size]
                results = self._process_batch(batch, study['inclusion_criteria'], study['exclusion_criteria'])

                if results:
                    for res in results:
                        sid = res.get('subject_id')
                        study_eligible_ids.append(sid)
                        audit_data.append({
                            "nct_id": nct_id,
                            "subject_id": sid,
                            "reason": res.get('reason')
                        })

                time.sleep(4) # Respeitando a cota da API

            # --- SALVAMENTO INCREMENTAL ---
            # Salva o relatório principal
            report_row = pd.DataFrame([{
                "nct_id": nct_id,
                "eligible_count": len(study_eligible_ids),
                "subject_ids": json.dumps(study_eligible_ids)
            }])
            report_row.to_csv(report_file, mode='a', header=not os.path.exists(report_file), index=False)

            # Salva o log de auditoria
            if audit_data:
                pd.DataFrame(audit_data).to_csv(log_file, mode='a', header=not os.path.exists(log_file), index=False)

            # Adiciona aos processados para controle em tempo de execução
            processed_ncts.add(nct_id)

        return pd.read_csv(report_file), pd.read_csv(log_file)

In [ ]:
## Main

In [ ]:
# @title Define Engine

engine = BatchClinicalMatch(
    api_key="ParsingGeminiAPI",
    version_genai = 'gemini-2.5-flash'
)

In [ ]:
# @title Subsets data
all_cols = mimic_patients_df.columns.tolist()

cols_obrigatorias = ['subject_id', 'gender', 'age', 'name_diag_principal', 'list_procedures', 'HPI', 'chief_complaint']
to_ignore = ['hadm_id']
cols_com_dados = mimic_patients_df.columns[mimic_patients_df.notna().sum() > 0].tolist()
filtered_cols = [c for c in cols_com_dados if c not in to_ignore and not c.endswith('_FLAG')]
cols_finais = list(set(cols_obrigatorias + filtered_cols))
patients_subset = mimic_patients_df[cols_finais].copy()

def trim_clinical_text(text, limit=600):
    text = str(text)
    if len(text) <= limit:
        return text

    # Pega os primeiros 300 e os últimos 300 caracteres
    half = limit // 2
    return text[:half] + "\n[...] [TEXT CUT] [...]\n" + text[-half:]

# Aplica no seu DataFrame
patients_subset['HPI'] = patients_subset['HPI'].apply(trim_clinical_text)

In [ ]:
# @title Run report
report_df, audit_df = engine.run_inference(
    clinical_trials_df.head(10), # Teste com 10 estudos primeiro
    patients_subset,
    batch_size = 50,
    report_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_report.csv",
    log_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/audit_log.csv"
)

Retomando progresso: 1 estudos já processados.


Estudos:   0%|          | 0/10 [00:00<?, ?it/s]

Estudos:  20%|██        | 2/10 [52:13<3:28:53, 1566.65s/it]

Estudos:  30%|███       | 3/10 [1:39:47<4:05:23, 2103.31s/it]

Estudos:  40%|████      | 4/10 [2:21:19<3:44:49, 2248.27s/it]

...

Estudos:  60%|██████    | 6/10 [3:30:37<2:30:55, 2263.97s/it]

Estudos:  70%|███████   | 7/10 [3:51:31<1:36:57, 1939.12s/it]

Estudos:  80%|████████  | 8/10 [4:50:06<1:21:10, 2435.08s/it]

Estudos:  90%|█████████ | 9/10 [6:03:09<50:39, 3039.35s/it]  

Estudos: 100%|██████████| 10/10 [7:17:32<00:00, 2625.27s/it]
